# Датасеты Hugging Face: проверка карточек и образцов

In [ ]:
from pathlib import Path
import sys, random
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'captcha_lab').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
random.seed(42)
import numpy as np
import pandas as pd
np.random.seed(42)
from captcha_lab.data import image_manifest, ensure_labels, crop_image, save_json
from captcha_lab.benchmark import run_benchmark, save_run, evaluate
print('Python:', sys.version.split()[0])

Ищем данные для дообучения, не объявляя синтетику эквивалентом Instagram.
Показываем ограниченные viewer-образцы из test, без скачивания многогигабайтных архивов.
Текущая ревизия сверяется с lock; при изменении нужен повторный просмотр.
Нельзя обучаться на этих test-образцах и затем выдавать результат за независимый benchmark.

In [ ]:
import json
lock = json.loads((ROOT/'research/hf_lock.json').read_text(encoding='utf-8'))
display(pd.DataFrame([dict(repo=k,**v) for k,v in lock['datasets'].items()]))

In [ ]:
from captcha_lab.hf_data import fetch_preview
samples = {}
for repo in ['project-sloth/captcha-images','Hansda/captcha-dataset']:
    samples[repo] = fetch_preview(repo,split='test',limit=16)
    print(repo,'скачано образцов:',len(samples[repo]))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(4,4,figsize=(14,8))
for ax, row in zip(axes.flat,samples['project-sloth/captcha-images'].itertuples()):
    with Image.open(row.path) as im: ax.imshow(im)
    ax.set_title(row.label); ax.axis('off')
fig.suptitle('project-sloth/captcha-images: первые 16 test-образцов, шестизначные цифровые строки')
plt.tight_layout(); plt.show()

Выводы исследования и ограничения лицензий — в `research/HUGGING_FACE.md`.
Для локальных изображений без проверенных меток точность не считается.
Установленные отличия: другой шрифт/искажения; у Hansda исключены 0 и 1;
у pymlex строки из четырёх цифр. Эти наборы не заменяют целевую контрольную выборку.

## Ограниченная проверка на известных метках

Первые 16 test-образцов цифрового набора: выбранная иллюстративная подвыборка, не весь test.
Модели могли обучаться на пересекающихся публичных данных; независимость этого набора от их обучения не установлена.
Результат нельзя переносить на Instagram. Изображения здесь уже вырезаны: `roi=None`.

In [ ]:
from captcha_lab.onnx_models import load_ddddocr, load_ppocr
from captcha_lab.hf_models import load_graf
public = samples['project-sloth/captcha-images']
factories = {'ddddocr-default':lambda:load_ddddocr(False),'ddddocr-beta':lambda:load_ddddocr(True),
             'ppocrv6-tiny':lambda:load_ppocr('tiny'),'ppocrv6-small':lambda:load_ppocr('small')}
for repo in ['Graf-J/captcha-crnn-base','Graf-J/captcha-crnn-finetuned',
             'Graf-J/captcha-conv-transformer-base','Graf-J/captcha-conv-transformer-finetuned']:
    factories[repo] = lambda repo=repo:load_graf(repo)
public_frames = []
for name, factory in factories.items():
    print('Public sample:',name,flush=True)
    predictor = factory()
    frame = run_benchmark(public,predictor,name,roi=None,repeats=3)
    frame.attrs['dataset'] = 'project-sloth/captcha-images'
    frame.attrs['dataset_revision'] = public.revision.iloc[0]
    frame.attrs['sample'] = 'first 16 test viewer rows; not representative; training overlap unknown'
    save_run(frame,'public/'+name.replace('/','--')+'.csv')
    public_frames.append(frame)
    del predictor
public_summary = evaluate(pd.concat(public_frames,ignore_index=True),public)
public_summary.to_csv(ROOT/'results/public/summary.csv',index=False)
display(public_summary)